In [16]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time
import random
import torch.nn.functional as F
import mujoco
import mujoco.viewer

In [17]:
#functions to convert between real and simulated robot values
def sim2real(state):
    x = state.copy()
    x[2] *= -1
    x[3], x[4] = x[4], x[3]
    x[6] *=2
    x = x[:-1]
    return x

def real2sim(state):
    x = state.copy()
    x[2] *= -1
    x[3], x[4] = x[4], x[3]
    x[6] /= 2
    x = np.append(x, x[6])
    return x

In [18]:
class CobotEnv(gym.Env):
    def __init__(self, render_mode = None):
        xml_path= "/home/aaron-dsouza/programming/digital_twin/robots/mobile_aloha_sim/aloha_mujoco/aloha/meshes_mujoco/aloha_v1.xml"
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_path(xml_path)
        self.data = mujoco.MjData(self.model)
        self.steps = 0
        max_delta_const = 0.1
        self.max_delta = np.array([2, 0.7, 0.7, 1, 1, 1, 0.1]) * max_delta_const
        self.max_speed = 8.0
        self.max_steps = 500
        self.viewer = None

        self.motor_qpos_low = np.array([-3.14158, 0, -3.14158, -2, -1.5708, -3.14158, 0])
        self.motor_qpos_high = np.array([3.14158, 3.14158, 0, 2, 1.5708, 3.14158, 0.095])

        self.motor_qvel_low = np.full(7, -8.0)
        self.motor_qvel_high = np.full(7, 8.0)
        
        self.cube_pos_low = np.array([0.5, -0.4, 0.7])
        self.cube_pos_high = np.array([0.8, 0.05, 1.5])
        
        self.disk_pos_low = np.array([-0.1, -0.55, 0.7])
        self.disk_pos_high = np.array([0.2, -0.1, 0.8])

        self.gripper_pos_low = np.array([0.3, -0.6, 0.75])
        self.gripper_pos_high = np.array([0.95, 0.2, 1.5])

        all_obs_low = np.concatenate((self.motor_qpos_low, self.motor_qvel_low, self.cube_pos_low, 
                                      self.disk_pos_low, self.gripper_pos_low), dtype=np.float32)
        all_obs_high = np.concatenate((self.motor_qpos_high, self.motor_qvel_high, self.cube_pos_high, 
                                       self.disk_pos_high, self.gripper_pos_high), dtype=np.float32)

        self.observation_space = spaces.Box(
            low=all_obs_low, high=all_obs_high, dtype=np.float32
        )

        self.action_space = spaces.Box(
            low=-self.max_delta, high=self.max_delta, shape=(7,), dtype=np.float32
        )

    def get_observation(self):
        motor_qpos = sim2real(self.data.qpos[-8:])
        motor_qvel = sim2real(self.data.qvel[-8:])
        cube_pos = self.data.xpos[1]
        disk_pos = self.data.xpos[2]
        gripper_pos = self.data.site_xpos[1]
        all_observations = np.concatenate((motor_qpos, motor_qvel, cube_pos, disk_pos, gripper_pos), dtype=np.float32)
        
        return all_observations

    def apply_action(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high)
        desired_pos = sim2real(self.data.qpos[-8:]) + action
        desired_pos = np.clip(desired_pos, self.motor_qpos_low, self.motor_qpos_high)
        
        self.data.ctrl[-8:] = real2sim(desired_pos)
        for _ in range(20):
            mujoco.mj_step(self.model, self.data)

    def compute_rewards(self):
        raise NotImplementedError

    def check_bounds(self, position, min_pos, max_pos):
        # positions = [self.data.qpos[:3], self.data.qpos[:3], self.data.site_xpos[1]]
        # min_pos = [self.cube_pos_low, self.disk_pos_low, self.gripper_pos_low]
        # max_pos = [self.cube_pos_high, self.disk_pos_high, self.gripper_pos_high]

        within_low = min_pos <= position
        within_high =position <= max_pos 
        is_inside = np.all(within_low) and np.all(within_high)

        return is_inside

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)

        cube_x = np.random.uniform(self.cube_pos_low[0], self.cube_pos_high[0])
        cube_y = np.random.uniform(self.cube_pos_low[1], self.cube_pos_high[1])
        self.data.qpos[:2] = np.array([cube_x, cube_y])

        disk_x = np.random.uniform(self.disk_pos_low[0], self.disk_pos_high[0])
        disk_y = np.random.uniform(self.disk_pos_low[1], self.disk_pos_high[1])
        self.data.qpos[7:9] = np.array([disk_x, disk_y])

        mujoco.mj_forward(self.model, self.data)
        self.steps = 0
        return self.get_observation(), {}
        
    def step(self, action):
        self.apply_action(action)
        next_state = self.get_observation()
        reward, info = self.compute_rewards()
        self.steps +=1

        is_inside = self.check_bounds(self.data.qpos[:3], self.cube_pos_low, self.gripper_pos_high)
        
        terminated = (
            np.max(np.abs(self.data.qacc[-8:])) > 1e6 or
            info["success"] or 
            not is_inside
        )
        truncated = self.steps >= self.max_steps

        if self.render_mode == 'human':
            self.render()
        
        #next_state, reward, terminated, truncated, info =
        return next_state, reward, terminated, truncated, info

    def render(self):
        if self.viewer == None:
            self.viewer = mujoco.viewer.launch_passive(self.model, self.data)

        self.viewer.sync()

        if not self.viewer.is_running():
            self.viewer.close()
            self.viewer = None

    def close(self):
        if self.viewer is not None:
            self.viewer.close()
            self.viewer = None

In [19]:
class CobotApproachEnv(CobotEnv):
    def __init__(self, render_mode = None):
        super().__init__(render_mode)

    def compute_rewards(self):
        gripper_pos = self.data.site_xpos[1]
        cube_pos = self.data.xpos[1]
        # approach_pos = cube_pos + np.array([0, 0, 0.1])
        cube_distance = np.linalg.norm((gripper_pos-cube_pos))
        # approach_distance = np.linalg.norm((gripper_pos-approach_pos))
        info = {'success': False}

        distance_reward = -5 * cube_distance
        # cube_reward = -10 * cube_distance # + 3.0
        # transition = np.clip((approach_distance - 0.1) / 0.1, 0, 1)
        # distance_reward = (
        #     transition * approach_reward +
        #     (1 - transition) * cube_reward
        # )

        #orientation reward
        gripper_rot = self.data.site_xmat[1].reshape(3, 3)[:, 0]
        z_axis = np.array([0, 0, -1.0])
        to_cube = np.array((cube_pos-gripper_pos)/ cube_distance)
        orientation_weight = 0.5 * max(0, (1 - cube_distance/0.4))**2
        orientation_reward = (
            np.dot(gripper_rot, z_axis) * 1 +
            np.dot(gripper_rot, to_cube) * 2.5
        ) * orientation_weight

        #joint penalty
        joint_margin = 2.8
        joint_penalty = (
            max (self.data.qpos[-7]-joint_margin, 0) + 
            max (self.data.qpos[-6]-joint_margin, 0)
        ) * 1.0
         
        # velocity_cost = 0.0001 #* np.sum(sim2real(self.data.qvel[-8:])**2)

        reward = distance_reward + orientation_reward - joint_penalty

        info = {'success': False}
        # print('d', distance, ':v', velocity_cost)
        gripper_inside = self.check_bounds(gripper_pos, self.gripper_pos_low, self.gripper_pos_high)
        if not gripper_inside:
            reward -= 0.5

        if cube_distance <=0.05:
            print('cube approach success')
            info["success"] = True
        
        return reward, info

In [20]:
# env = CobotApproachEnv(render_mode='human')
# env.render()
# episodeNumber = 10
# timesteps = 100
# for episodeIndex in range(episodeNumber):
#     initial_state= env.reset()
#     for timeIndex in range(timesteps):
#         rand_action = env.action_space.sample()
#         observation, reward, terminated, truncated, info = env.step(rand_action)
#         print(reward)
#         time.sleep(0.1)
#         if(terminated or truncated):
#             time.sleep(1)
#             break

# env.close()

In [21]:
from collections import deque

class ReplayBuffer:
    def __init__(self, buffer_size=50000):
        self.buffer = deque(maxlen=buffer_size)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.stack, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [22]:
class CriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()
        self.critic1= nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1)
        )
        self.critic2 = nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, state, action):
        inp = torch.cat([state, action], dim=-1)
        Q1 = self.critic1(inp)
        Q2 = self.critic2(inp)
        return Q1, Q2

In [23]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims, action_limit):
        super().__init__()
        self.feature_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                )
        self.mean_head = nn.Linear(hidden_dims[1], action_dim)
        self.log_std_head = nn.Linear(hidden_dims[1], action_dim)
        self.register_buffer(
            "action_limit",
            torch.tensor(action_limit, dtype=torch.float32)
        )

    def forward(self, state):
        features = self.feature_head(state)
        mean = self.mean_head(features)
        log_std = self.log_std_head(features)

        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        sample = dist.rsample()
        action = torch.tanh(sample) 

        log_prob = dist.log_prob(sample)
        log_prob -= torch.log(1-action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        action = action * self.action_limit
    
        return action, log_prob

    def sample(self, state):
        features = self.feature_head(state)
        mean = self.mean_head(features)
        action = torch.tanh(mean)
        action = action * self.action_limit
        return action

In [ ]:
class SoftActorCritic:
    def __init__(self, render_mode=None):
        self.env = CobotApproachEnv(render_mode)

        state_dim = 7+7+3+3+3
        action_dim = 7
        hidden_dim = (256, 256)

        self.critic = CriticNetwork(state_dim, action_dim, hidden_dim)
        self.actor = ActorNetwork(state_dim, action_dim, hidden_dim, 
                                  self.env.max_delta)

        self.target_entropy = -action_dim 
        self.target_critic = CriticNetwork(state_dim, action_dim, hidden_dim)
        self.target_critic.load_state_dict(self.critic.state_dict())
        for param in self.target_critic.parameters():
            param.requires_grad = False

        initial_alpha = 0.2
        
        self.log_alpha = torch.tensor(np.log(initial_alpha), dtype=torch.float32, 
                                      requires_grad=True)

        self.replay_buffer = ReplayBuffer(200_000)

        self.gamma = 0.98
        self.batch_size = 256

        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=4e-4)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=4e-4)
        self.alpha_optimizer = torch.optim.Adam([self.log_alpha], lr=3e-4)

    @torch.no_grad()
    def get_target(self, next_states, rewards, dones):
        next_states = torch.as_tensor(next_states, dtype=torch.float32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.as_tensor(dones, dtype=torch.float32).unsqueeze(1)
        next_actions, log_prob = self.actor(next_states)

        target_Q1, target_Q2 = self.target_critic(next_states, next_actions)
        target_Q = torch.min(target_Q1, target_Q2)
        target = rewards + self.gamma * (1-dones) * (
            target_Q - self.log_alpha.exp() * log_prob
        )
        return target

    def update_critic(self, targets, states, actions):
        states = torch.as_tensor(states, dtype=torch.float32)
        actions = torch.as_tensor(actions, dtype=torch.float32)

        Q1, Q2 = self.critic(states, actions)

        critic_loss = F.mse_loss(Q1, targets) + F.mse_loss(Q2, targets)   

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

    def update_actor(self, states):
        states = torch.as_tensor(states, dtype=torch.float32)

        new_actions, log_prob = self.actor(states)
        Q1, Q2 = self.critic(states, new_actions)
        Q = torch.min(Q1, Q2)

        actor_loss = (self.log_alpha.exp() * log_prob - Q).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
        self.alpha_optimizer.zero_grad()
        alpha_loss.backward()
        self.alpha_optimizer.step()


    def soft_target_update(self, tau=0.005):
        for target_param, critic_param in zip(self.target_critic.parameters(), 
                                            self.critic.parameters()):
            target_param.data.copy_(
                tau * critic_param.data +
                (1-tau) * target_param.data
            )

    def train(self, n_iterations, resume=False):
        self.actor.train()
        self.critic.train()
        state, _ = self.env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        episode_reward = 0
        for step in range(n_iterations):
            if step < n_iterations//10 and not resume:
                action = self.env.action_space.sample()
            else:
                action, _ = self.actor(state)
                action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            # print('action',action)
            # print('ctrl',self.env.data.ctrl)
            # print('qpo',self.env.data.qpos[-8:])
            episode_reward += reward
            done = terminated or truncated
            self.replay_buffer.push(state.squeeze(0).numpy(), action, reward, next_state, done)

            if len(self.replay_buffer) >= self.batch_size:
                states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
                
                targets = self.get_target(next_states, rewards, dones)
                self.update_critic(targets, states, actions)
                self.update_actor(states)
                self.soft_target_update()

            if done:    
                print(f"Step {step}: {episode_reward:.2f}")
                episode_reward = 0
                state, _ = self.env.reset()
                state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
                
            else:
                state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

    @torch.no_grad()
    def eval(self):
        self.actor.eval()
        eval_env = CobotApproachEnv(render_mode="human")
        state, _ = eval_env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        done = False
        while(not done):
            action = self.actor.sample(state)
            action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)
            time.sleep(0.01)
        eval_env.close()

: 

In [ ]:
# sac = SoftActorCritic()

# note: use sac_checkpoinrt_robot_arm_1.pt for latest model, but unstable
#     or use arm.pt for stable but only approaches top
sac = SoftActorCritic(render_mode="human")
# checkpoint = torch.load("sac_checkpoint_robot_arm.pt", map_location=torch.device('cpu')) # Use 'cpu' to safely load on any machine

# sac.actor.load_state_dict(checkpoint["actor"])
# sac.critic.load_state_dict(checkpoint["critic"])
# sac.target_critic.load_state_dict(checkpoint["target_critic"])

# sac.actor_optimizer.load_state_dict(checkpoint["actor_optimizer"])
# sac.critic_optimizer.load_state_dict(checkpoint["critic_optimizer"])
sac.train(40_000, resume=False)

Step 499: -816.68


In [ ]:
# sac.train(70)#_000, resume=True)

In [ ]:
for i in range(2):
    sac.eval()

In [ ]:
# torch.save({
#     "actor": sac.actor.state_dict(),
#     "critic": sac.critic.state_dict(),
#     "target_critic": sac.target_critic.state_dict(),
#     "actor_optimizer": sac.actor_optimizer.state_dict(),
#     "critic_optimizer": sac.critic_optimizer.state_dict(),
# }, "sac_checkpoint_robot_arm_1.pt")

In [ ]:
sac.env.render()